In [1]:
import torch

In [2]:
torch.cuda.get_device_name(0)

'NVIDIA GeForce GTX 1080'

In [3]:
torch.cuda.is_available()

True

In [4]:
import torch



# Dataset 

In [6]:

import numpy as np
import glob
import time
import cv2



def load_file(filename):
    arrays = np.load(filename)
    #arrays = np.stack([cv2.cvtColor(arrays[_], cv2.COLOR_BGR2GRAY)
     #                 for _ in range(30)], axis=0)
    #arrays = arrays / 255.
    return arrays

In [7]:
class MyDataset():
    def __init__(self, folds, path):
        self.folds = folds
        self.path = path
        with open(r'C:\Users\ml\Desktop\extention all\Full LRW\lrw.txt') as myfile:
            lines = myfile.read().splitlines()
        self.data_dir = [self.path + item for item in lines]
        self.data_files = glob.glob(self.path+'*/'+self.folds+'/*.npy')
        self.labels = {}
        for idx, label in enumerate(list(open(r'C:\Users\ml\Desktop\extention all\Full LRW\lrw.txt'))):
            label = label.strip()
            self.labels[label] = idx

    def __getitem__(self, idx):
        inputs = load_file(self.data_files[idx])
        labels = self.labels[self.data_files[idx].split('\\')[-1].split('_')[0]]
        #print("Label from file path:", labels)
        return inputs, labels

    def __len__(self):
        return len(self.data_files)


# cvtransforms

In [8]:
import random
import cv2
import numpy as np


def CenterCrop(batch_img, size):
    w, h = batch_img[0][0].shape[1], batch_img[0][0].shape[0]
    th, tw = size
    img = np.zeros((len(batch_img), len(batch_img[0]), th, tw))
    for i in range(len(batch_img)):
        x1 = int(round((w - tw))/2.)
        y1 = int(round((h - th))/2.)
        img[i] = batch_img[i, :, y1:y1+th, x1:x1+tw]
    return img

In [9]:


def RandomCrop(batch_img, size):
    w, h = batch_img[0][0].shape[1], batch_img[0][0].shape[0]
    th, tw = size
    img = np.zeros((len(batch_img), len(batch_img[0]), th, tw))
    for i in range(len(batch_img)):
        x1 = random.randint(0, 8)
        y1 = random.randint(0, 8)
        img[i] = batch_img[i, :, y1:y1+th, x1:x1+tw]
    return img

In [10]:

def HorizontalFlip(batch_img):
    for i in range(len(batch_img)):
        if random.random() > 0.5:
            for j in range(len(batch_img[i])):
                batch_img[i][j] = cv2.flip(batch_img[i][j], 1)
    return batch_img


def ColorNormalize(batch_img):
    mean = 0.413621
    std = 0.1700239
    batch_img = (batch_img - mean) / std
    return batch_img



# Model

In [11]:

import math
import numpy as np


import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd import Variable


def conv3x3(in_planes, out_planes, stride=1):
    return nn.Conv2d(in_planes, out_planes, kernel_size=3, stride=stride,
                     padding=1, bias=False)

In [12]:
class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, inplanes, planes, stride=1, downsample=None):
        super(BasicBlock, self).__init__()
        self.conv1 = conv3x3(inplanes, planes, stride)
        self.bn1 = nn.BatchNorm2d(planes)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = conv3x3(planes, planes)
        self.bn2 = nn.BatchNorm2d(planes)
        self.downsample = downsample
        self.stride = stride

    def forward(self, x):
        residual = x
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.bn2(out)
        if self.downsample is not None:
            residual = self.downsample(x)

        out += residual
        out = self.relu(out)

        return out

In [13]:
class ExceptionModel(nn.Module):

    def __init__(self, block, layers, num_classes=50):
        self.inplanes = 64
        super(ExceptionModel, self).__init__()
        self.layer1 = self._make_layer(block, 64, layers[0])
        self.layer2 = self._make_layer(block, 128, layers[1], stride=2)
        self.layer3 = self._make_layer(block, 256, layers[2], stride=2)
        self.layer4 = self._make_layer(block, 512, layers[3], stride=2)
        self.gloablavgpool = nn.AdaptiveAvgPool2d((1,1))
        self.bn = nn.BatchNorm1d(512)
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                n = m.kernel_size[0] * m.kernel_size[1] * m.out_channels
                m.weight.data.normal_(0, math.sqrt(2. / n))
            elif isinstance(m, nn.BatchNorm2d):
                m.weight.data.fill_(1)
                m.bias.data.zero_()
            elif isinstance(m, nn.BatchNorm1d):
                m.weight.data.fill_(1)
                m.bias.data.zero_()

    def _make_layer(self, block, planes, blocks, stride=1):
        downsample = None
        if stride != 1 or self.inplanes != planes * block.expansion:
            downsample = nn.Sequential(
                nn.Conv2d(self.inplanes, planes * block.expansion,
                          kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(planes * block.expansion),
            )

        layers = []
        layers.append(block(self.inplanes, planes, stride, downsample))
        self.inplanes = planes * block.expansion
        for i in range(1, blocks):
            layers.append(block(self.inplanes, planes))

        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.gloablavgpool(x)
        
        
        
        x= x.view(x.size(0),-1)
        x = self.bn(x)



        return x

In [14]:
class GRU(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, num_classes, every_frame=True):
        super(GRU, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.every_frame = every_frame
        self.gru = nn.GRU(input_size, hidden_size, num_layers, batch_first=True, bidirectional=True, dropout=0.2)
        self.fc = nn.Linear(hidden_size*2, num_classes)
        self.dropout = nn.Dropout(0.2)

    def forward(self, x):
        self.gru.flatten_parameters()
        h0 = Variable(torch.zeros(self.num_layers*2, x.size(0), self.hidden_size))
        out, _ = self.gru(x, h0)
        
        out = self.dropout(out).mean(1)
        out = self.fc(out)

        return out

In [15]:
class Lipreading(nn.Module):
    def __init__(self, mode, inputDim=256, hiddenDim=1024, nClasses=50, frameLen=30, every_frame=True):
        super(Lipreading, self).__init__()
        self.mode = mode
        self.inputDim = inputDim
        self.hiddenDim = hiddenDim
        self.nClasses = nClasses
        self.frameLen = 30
        self.every_frame = every_frame
        self.nLayers = 3
        # frontend3D
        self.frontend3D = nn.Sequential(
                nn.Conv3d(1, 64, kernel_size=(5, 7, 7), stride=(1, 2, 2), padding=(2, 3, 3), bias=False),
                nn.BatchNorm3d(64),
                nn.ReLU(True),
                nn.MaxPool3d(kernel_size=(1, 3, 3), stride=(1, 2, 2), padding=(0, 1, 1))
                )
        

        self.ExceptionModel = ExceptionModel(BasicBlock, [2, 2, 2, 2], num_classes=self.inputDim)
        self.dropout = nn.Dropout(0.2)



        self.gru = GRU(self.inputDim, self.hiddenDim, self.nLayers, self.nClasses, self.every_frame)
            # initialize
        self._initialize_weights()

    def forward(self, x):
        x = self.frontend3D(x)
        x = x.transpose(1, 2)
        x = x.contiguous()
        x = x.view(-1, 64, x.size(3), x.size(4))
        x = self.ExceptionModel(x)

        x = self.dropout(x)

        x = x.view(-1, self.frameLen, self.inputDim)
        x = self.gru(x)

        return x

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv3d):
                n = m.kernel_size[0] * m.kernel_size[1] * m.kernel_size[2] * m.out_channels
                m.weight.data.normal_(0, math.sqrt(2. / n))
                if m.bias is not None:
                    m.bias.data.zero_()

            elif isinstance(m, nn.Conv2d):
                n = m.kernel_size[0] * m.kernel_size[1] * m.out_channels
                m.weight.data.normal_(0, math.sqrt(2. / n))
                if m.bias is not None:
                    m.bias.data.zero_()

            elif isinstance(m, nn.Conv1d):
                n = m.kernel_size[0] * m.out_channels
                m.weight.data.normal_(0, math.sqrt(2. / n))
                if m.bias is not None:
                    m.bias.data.zero_()

            elif isinstance(m, nn.BatchNorm3d):
                m.weight.data.fill_(1)
                m.bias.data.zero_()

            elif isinstance(m, nn.BatchNorm2d):
                m.weight.data.fill_(1)
                m.bias.data.zero_()

            elif isinstance(m, nn.BatchNorm1d):
                m.weight.data.fill_(1)
                m.bias.data.zero_()

In [16]:

!pip install torchsummary

In [17]:

if (__name__ == '__main__'):
    import torchsummary
    model = ExceptionModel(BasicBlock, [3,4,6,3], num_classes=50)
    #torchsummary.summary(model,(64,22,22),device='cpu')

# Main

In [18]:
import os
import time
import random
import logging
import argparse
import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

#from model import *
#from dataset import *
#from cvtransforms import *

SEED = 1
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
np.random.seed(SEED)

In [19]:
import numpy as np
import torch
import torchvision
from torchvision import datasets, models, transforms
import torch.utils.data as data
#from torch.utils.tensorboard import SummaryWriter
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
#from nets import *
import time, os, copy, argparse
import multiprocessing
from torchsummary import summary
#from matplotlib import pyplot as plt

C:\Users\ml\.conda\envs\py39\lib\site-packages\torchvision\io\image.py:13: UserWarning: Failed to load image Python extension: '[WinError 127] The specified procedure could not be found'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(


In [20]:
parser = argparse.ArgumentParser(description='Pytorch-GLMIM-LRW')
parser.add_argument('--nClasses', default=50, type=int, help='the number of classes')
#path : path to model, when you start Baseline, it's empty, otherwise it would load the model.
parser.add_argument('--path', default=r'', type=str, help='path to Baseline, empty for training')
parser.add_argument('--dataset', default=r'C:/Users/ml/Desktop/extention all/Full LRW/', type=str, help='path/to/lrw/roi_80_116_175_211_npy_gray/')
parser.add_argument('--mode', default='Baseline', type=str)
parser.add_argument('--every-frame', default=True, action='store_false', help='predicition based on every frame')
parser.add_argument('--lr', default=1e-4, type=float, help='initial learning rate')
parser.add_argument('--batch-size', default=32, type=int, help='mini-batch size')
#parser.add_argument('--workers', default=5, type=int, help='number of data loading workers')
parser.add_argument('--epochs', default=75, type=int, help='number of total epochs')
parser.add_argument('--s_epochs', default=1, type=int, help='number of start epochs')
parser.add_argument('--interval', default=40, type=int, help='display interval')
parser.add_argument('--test', default=False, action='store_false', help='perform on the test phase')


_StoreFalseAction(option_strings=['--test'], dest='test', nargs=0, const=False, default=False, type=None, choices=None, required=False, help='perform on the test phase', metavar=None)

In [21]:

args = parser.parse_args([])


In [22]:
torch.cuda.is_available()
if torch.cuda.is_available():
    torch.set_default_tensor_type('torch.cuda.FloatTensor')

C:\Users\ml\.conda\envs\py39\lib\site-packages\torch\__init__.py:614: UserWarning: torch.set_default_tensor_type() is deprecated as of PyTorch 2.1, please use torch.set_default_dtype() and torch.set_default_device() as alternatives. (Triggered internally at C:\cb\pytorch_1000000000000\work\torch\csrc\tensor\python_tensor.cpp:453.)
  _C._set_default_tensor_type(t)


In [23]:


def data_loader(args):
    dsets = {x: MyDataset(x, args.dataset) for x in ['train','val', 'test']}
    dset_loaders = {x: DataLoader(dsets[x], batch_size=args.batch_size, shuffle=True,generator=torch.Generator(device="cuda")) for x in ['train', 'val', 'test']}
    dset_sizes = {x: len(dsets[x]) for x in ['train', 'val', 'test']}
    #print('\nStatistics: train: {}, val: {}, test: {}'.format(dset_sizes['train'], dset_sizes['val'], dset_sizes['test']))
    return dset_loaders, dset_sizes

In [24]:


def reload_model(model, logger, path=""):
    if not bool(path):
        logger.info('train from scratch')
        return
    own_state = model.state_dict()
    state_dict = torch.load(path)
    for name, param in state_dict.items():
        if name not in own_state:
            print('layer {} skip, not exist'.format(name))
            continue
        if isinstance(param, nn.Parameter):
            param = param.data
        if own_state[name].shape != param.shape:
            print('layer {} skip, shape not same'.format(name))
            continue
        own_state[name].copy_(param)

In [25]:
def showLR(optimizer):
    lr = []
    for param_group in optimizer.param_groups:
        lr += [param_group['lr']]
    return lr

<h1> new test, train, test_adam

In [26]:
import time
import numpy as np


def train(model, dset_loaders, criterion, epoch, phase, optimizer, args, logger, use_gpu):
    model.train()
    logger.info('-' * 2)
    logger.info('Epoch {}/{}'.format(epoch, args.epochs - 1))
    logger.info('Current Learning rate: {}'.format(showLR(optimizer)))

    running_loss, running_corrects, running_all = 0., 0., 0.
    since = time.time()
    last_time_batch_idx = -1
    for batch_idx, (inputs, targets) in enumerate(dset_loaders[phase]):
        batch_img = RandomCrop(inputs.cpu().numpy(), (90, 90))
        batch_img = ColorNormalize(batch_img)
        batch_img = HorizontalFlip(batch_img)

        batch_img = np.reshape(batch_img, (batch_img.shape[0], batch_img.shape[1], batch_img.shape[2], batch_img.shape[3], 1))
        inputs = torch.from_numpy(batch_img)
        inputs = inputs.float().permute(0, 4, 1, 2, 3).contiguous()
        if use_gpu:
            inputs = inputs.cuda()
            targets = targets.cuda()
        optimizer.zero_grad()
        outputs = model(inputs)
        _, preds = torch.max(outputs.data, 1)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        # statistics
        running_loss += loss.item() * inputs.size(0)
        batch_correct = (preds == targets.data).sum().item()
        running_corrects += batch_correct
        running_all += len(inputs)

        if batch_idx % args.interval == 0 or (batch_idx == len(dset_loaders[phase])-1):
            logger.info('Process:  [({:.0f}%)]\tLoss batch: {:.4f}\tLoss total: {:.4f}\tAcc batch:{:.4f}\tAcc total:{:.4f}\tEstimated time:{:5.0f}s'.format(
             
                100. * batch_idx / (len(dset_loaders[phase])-1),
                float(loss),
                float(running_loss) / running_all,
                float(batch_correct) / len(inputs),
                float(running_corrects) / running_all,
                (time.time() - since) / (batch_idx - last_time_batch_idx) * (len(dset_loaders[phase]) - batch_idx - 1))),
            last_time_batch_idx = batch_idx
            since = time.time()

    loss_epoch =  float(running_loss) / len(dset_loaders[phase].dataset)
    acc_epoch = float(running_corrects) / len(dset_loaders[phase].dataset)

    logger.info('{} Epoch:\t{:2}\tLoss: {:.4f}\tAcc:{:.4f}\n'.format(
        phase,
        epoch,
        loss_epoch,
        acc_epoch))
    
    return loss_epoch, acc_epoch

In [27]:
def test2(model, dset_loaders, criterion, epoch, phase, args, logger, use_gpu, save=True):
    model.eval()
    with torch.no_grad():
        running_loss, running_corrects, running_all = 0., 0., 0.
        since = time.time()
        last_time_batch_idx = -1
        all_preds = []
        all_labels = []
        for batch_idx, (inputs, targets) in enumerate(dset_loaders[phase]):
            batch_img = CenterCrop(inputs.cpu().numpy(), (90, 90))
            batch_img = ColorNormalize(batch_img)

            batch_img = np.reshape(batch_img, (batch_img.shape[0], batch_img.shape[1], batch_img.shape[2], batch_img.shape[3], 1))
            inputs = torch.from_numpy(batch_img)
            inputs = inputs.float().permute(0, 4, 1, 2, 3).contiguous()
            if use_gpu:
                inputs = inputs.cuda()
                targets = targets.cuda()
            outputs = model(inputs)

            _, preds = torch.max(outputs.data, 1)
            loss = criterion(outputs, targets)
            # statistics
            running_loss += loss.item() * inputs.size(0)
            running_corrects += (preds == targets.data).sum().item()
            running_all += len(inputs)

            if batch_idx % args.interval == 0 or (batch_idx == len(dset_loaders[phase])-1):
                logger.info('Process: [({:.0f}%)]\tLoss: {:.4f}\tAcc:{:.4f}\tEstimated time:{:5.0f}s'.format(
                    
                    100. * batch_idx / (len(dset_loaders[phase])-1),
                    float(running_loss) / running_all,
                    float(running_corrects) / running_all,
                    (time.time() - since)/(batch_idx-last_time_batch_idx) * (len(dset_loaders[phase]) - batch_idx - 1))),
                last_time_batch_idx = batch_idx
                since = time.time()

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(targets.cpu().numpy())

        loss_epoch = float(running_loss) / len(dset_loaders[phase].dataset)
        acc_epoch = float(running_corrects) / len(dset_loaders[phase].dataset)
        logger.info('{} Epoch:\t{:2}\tLoss: {:.4f}\tAcc:{:.4f}'.format(
            phase,
            epoch,
            loss_epoch,
            acc_epoch)+'\n')

    if save:
        # Returning test accuracy and confusion matrix along with other necessary values
        return loss_epoch, acc_epoch, all_preds, all_labels
    else:
        return None, None, None, None  # Return placeholders for consistency


In [28]:
def test(model, dset_loaders, criterion, epoch, phase, args, logger, use_gpu, save=True):
    model.eval()
    with torch.no_grad():
        running_loss, running_corrects, running_all = 0., 0., 0.
        since = time.time()
        last_time_batch_idx = -1
        all_preds = []
        all_labels = []
        for batch_idx, (inputs, targets) in enumerate(dset_loaders[phase]):
            batch_img = CenterCrop(inputs.cpu().numpy(), (90, 90))
            batch_img = ColorNormalize(batch_img)

            batch_img = np.reshape(batch_img, (batch_img.shape[0], batch_img.shape[1], batch_img.shape[2], batch_img.shape[3], 1))
            inputs = torch.from_numpy(batch_img)
            inputs = inputs.float().permute(0, 4, 1, 2, 3).contiguous()
            if use_gpu:
                inputs = inputs.cuda()
                targets = targets.cuda()
            outputs = model(inputs)

            _, preds = torch.max(outputs.data, 1)
            loss = criterion(outputs, targets)
            # statistics
            running_loss += loss.item() * inputs.size(0)
            running_corrects += (preds == targets.data).sum().item()
            running_all += len(inputs)

            if batch_idx % args.interval == 0 or (batch_idx == len(dset_loaders[phase])-1):
                logger.info('Process:  [({:.0f}%)]\tLoss: {:.4f}\tAcc:{:.4f}\tEstimated time:{:5.0f}s'.format(
                   
                    100. * batch_idx / (len(dset_loaders[phase])-1),
                    float(running_loss) / running_all,
                    float(running_corrects) / running_all,
                    (time.time() - since)/(batch_idx-last_time_batch_idx) * (len(dset_loaders[phase]) - batch_idx - 1))),
                last_time_batch_idx = batch_idx
                since = time.time()

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(targets.cpu().numpy())

        loss_epoch = float(running_loss) / len(dset_loaders[phase].dataset)
        acc_epoch = float(running_corrects) / len(dset_loaders[phase].dataset)
        logger.info('{} Epoch:\t{:2}\tLoss: {:.4f}\tAcc:{:.4f}'.format(
            phase,
            epoch,
            loss_epoch,
            acc_epoch)+'\n')

    if save:
        # Returning test accuracy and confusion matrix along with other necessary values
        return loss_epoch, acc_epoch
    else:
        return None, None, None, None  # Return placeholders for consistency


In [29]:
import os
import logging
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score

def test_adam(args, use_gpu):
    save_path = './' + args.mode
    if not os.path.isdir(save_path):
        os.mkdir(save_path)
    
    # Logging setup
    filename = save_path + '/' + args.mode + '_' + str(args.lr) + '.txt'
    logger_name = "mylog"
    logger = logging.getLogger(logger_name)
    logger.setLevel(logging.INFO)
    fh = logging.FileHandler(filename, mode='a')
    fh.setLevel(logging.INFO)
    logger.addHandler(fh)
    console = logging.StreamHandler()
    console.setLevel(logging.INFO)
    logger.addHandler(console)

    # Model initialization and setup
    net = Lipreading(mode=args.mode, inputDim=512, hiddenDim=1024, nClasses=args.nClasses, frameLen=30, every_frame=args.every_frame)
    print(net)
    reload_model(net, logger, args.path)
    model = torch.nn.DataParallel(net)
    if use_gpu:
        model = model.cuda()
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(net.parameters(), lr=args.lr, weight_decay=5e-5, amsgrad=True)
    dset_loaders, dset_sizes = data_loader(args)

    train_losses = []
    val_losses = []
    train_accuracies = []
    val_accuracies = []
    best_val_acc = -1
    best_val_epoch = None
    best_test_acc = -1
    best_test_epoch = None
    best_confusion_matrix = None

    for epoch in range(args.s_epochs, args.epochs):
        train_loss, train_acc = train(model, dset_loaders, criterion, epoch, 'train', optimizer, args, logger, use_gpu)
        val_loss, val_acc = test(model, dset_loaders, criterion, epoch, 'val', args, logger, use_gpu, True)
        
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        train_accuracies.append(train_acc)
        val_accuracies.append(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_val_epoch = epoch

            state_dict = net.state_dict()

            _,best_test_acc, best_test_predictions, best_test_labels = test2(model, dset_loaders, criterion, epoch, 'test', args, logger, use_gpu, True)
            best_confusion_matrix = confusion_matrix(best_test_labels, best_test_predictions)
            best_test_epoch = epoch

    torch.save(state_dict, 'modelLRW10.pth')
    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    plt.plot(train_losses, label='Train Loss')
    plt.plot(val_losses, label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(train_accuracies, label='Train Accuracy')
    plt.plot(val_accuracies, label='Validation Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()

    plt.show()

    print("Best Validation Epoch:", best_val_epoch, "with Validation Accuracy:", best_val_acc)
    print("Best Test Epoch:", best_test_epoch, "with Test Accuracy:", best_test_acc)
    print("Confusion Matrix of the Best Model:\n", best_confusion_matrix)
    
    # Compute precision, recall, and F1 score
    precision = precision_score(best_test_labels, best_test_predictions, average='weighted')
    recall = recall_score(best_test_labels, best_test_predictions, average='weighted')
    f1 = f1_score(best_test_labels, best_test_predictions, average='weighted')

    # Print precision, recall, and F1 score
    print("Precision:", precision)
    print("Recall:", recall)
    print("F1 Score:", f1)
    
    # Plot confusion matrix
    sns.heatmap(best_confusion_matrix, annot=True)
    plt.xlabel('Predicted labels')
    plt.ylabel('True labels')
    plt.show()
    
    return best_test_acc, best_confusion_matrix
 


In [ ]:
if __name__ == '__main__':
    use_gpu = torch.cuda.is_available()
    test_adam(args, use_gpu)
    

train from scratch


Lipreading(
  (frontend3D): Sequential(
    (0): Conv3d(1, 64, kernel_size=(5, 7, 7), stride=(1, 2, 2), padding=(2, 3, 3), bias=False)
    (1): BatchNorm3d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool3d(kernel_size=(1, 3, 3), stride=(1, 2, 2), padding=(0, 1, 1), dilation=1, ceil_mode=False)
  )
  (ExceptionModel): ExceptionModel(
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (b

--
Epoch 1/74
Current Learning rate: [0.0001]
Process:  [(0%)]	Loss batch: 3.9205	Loss total: 3.9205	Acc batch:0.0000	Acc total:0.0000	Estimated time: 8391s
Process:  [(6%)]	Loss batch: 3.8115	Loss total: 3.9360	Acc batch:0.0938	Acc total:0.0183	Estimated time: 2417s
Process:  [(13%)]	Loss batch: 4.0058	Loss total: 3.9325	Acc batch:0.0312	Acc total:0.0204	Estimated time: 1532s
Process:  [(19%)]	Loss batch: 3.8706	Loss total: 3.9198	Acc batch:0.0312	Acc total:0.0243	Estimated time: 1504s
Process:  [(26%)]	Loss batch: 3.8512	Loss total: 3.9151	Acc batch:0.0000	Acc total:0.0266	Estimated time: 1345s
Process:  [(32%)]	Loss batch: 3.8921	Loss total: 3.9150	Acc batch:0.0312	Acc total:0.0297	Estimated time: 1273s
Process:  [(39%)]	Loss batch: 3.9030	Loss total: 3.9119	Acc batch:0.0000	Acc total:0.0296	Estimated time: 1142s
Process:  [(45%)]	Loss batch: 3.8804	Loss total: 3.9089	Acc batch:0.0000	Acc total:0.0295	Estimated time: 1019s
Process:  [(51%)]	Loss batch: 3.8266	Loss total: 3.9062	Acc 

In [ ]:
if __name__ == '__main__':
    use_gpu = torch.cuda.is_available()
    test_adam(args, use_gpu)
    

train from scratch
train from scratch
train from scratch
train from scratch
--
--
--
--
Epoch 1/74
Epoch 1/74
Epoch 1/74
Epoch 1/74
Current Learning rate: [0.0001]
Current Learning rate: [0.0001]
Current Learning rate: [0.0001]
Current Learning rate: [0.0001]


Lipreading(
  (frontend3D): Sequential(
    (0): Conv3d(1, 64, kernel_size=(5, 7, 7), stride=(1, 2, 2), padding=(2, 3, 3), bias=False)
    (1): BatchNorm3d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool3d(kernel_size=(1, 3, 3), stride=(1, 2, 2), padding=(0, 1, 1), dilation=1, ceil_mode=False)
  )
  (ExceptionModel): ExceptionModel(
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (b

Process:  [(0%)]	Loss batch: 2.3059	Loss total: 2.3059	Acc batch:0.0625	Acc total:0.0625	Estimated time: 2097s
Process:  [(0%)]	Loss batch: 2.3059	Loss total: 2.3059	Acc batch:0.0625	Acc total:0.0625	Estimated time: 2097s
Process:  [(0%)]	Loss batch: 2.3059	Loss total: 2.3059	Acc batch:0.0625	Acc total:0.0625	Estimated time: 2097s
Process:  [(0%)]	Loss batch: 2.3059	Loss total: 2.3059	Acc batch:0.0625	Acc total:0.0625	Estimated time: 2097s
Process:  [(36%)]	Loss batch: 2.2225	Loss total: 2.3149	Acc batch:0.1562	Acc total:0.1334	Estimated time: 1489s
Process:  [(36%)]	Loss batch: 2.2225	Loss total: 2.3149	Acc batch:0.1562	Acc total:0.1334	Estimated time: 1489s
Process:  [(36%)]	Loss batch: 2.2225	Loss total: 2.3149	Acc batch:0.1562	Acc total:0.1334	Estimated time: 1489s
Process:  [(36%)]	Loss batch: 2.2225	Loss total: 2.3149	Acc batch:0.1562	Acc total:0.1334	Estimated time: 1489s
Process:  [(73%)]	Loss batch: 2.2205	Loss total: 2.3119	Acc batch:0.1562	Acc total:0.1300	Estimated time:  5

<h1>check epoch 26

<h1> cache clear

In [ ]:
import gc
import torch
gc.collect()
torch.cuda.empty_cache()

In [ ]:
del model
gc.collect()
torch.cuda.empty_cache()